In [ ]:
# 사전설치 라이브러리 : pip install langgraph langchain-google-genai
import os
from typing import TypedDict, Annotated, List
import gradio as gr
from datetime import datetime
from fpdf import FPDF

from langchain_core.messages import BaseMessage, HumanMessage, AIMessage
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.graph import StateGraph, END

c:\aisource\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# --- 1. 환경 설정 및 API 키 설정 ---
os.environ["GOOGLE_API_KEY"] = "insert your key"

# --- 사전 준비: 폰트 파일 경로 설정 ---
FONT_PATH = "font/NanumGothic.ttf"

In [ ]:
# --- 2. LLM 초기화 ---
# temperature를 조절하여 답변의 창의성을 제어합니다.
llm = ChatGoogleGenerativeAI(model="gemini-2.0-flash", temperature=0.7)

In [ ]:
# --- 3. 그래프의 상태(State) 정의 ---
class AgentState(TypedDict):
    messages: Annotated[List[BaseMessage], lambda x, y: x + y]
    topic: str
    report_content: str # PDF 보고서 내용을 저장할 상태

# --- 헬퍼 함수: PDF 생성 ---
def create_pdf_report(content: str, filename: str):
    """주어진 텍스트 내용으로 PDF 파일을 생성합니다."""
    pdf = FPDF()
    pdf.add_page()
    try:
        pdf.add_font("NanumGothic", "", FONT_PATH, uni=True)
        pdf.set_font("NanumGothic", size=11)
    except RuntimeError:
        print(f"!!! 경고: '{FONT_PATH}' 폰트 파일을 찾을 수 없습니다.")
        pdf.set_font("Arial", size=11)
    pdf.multi_cell(0, 10, content)
    pdf.output(filename)
    print(f"--- PDF 저장 완료: {filename} ---")

In [ ]:
# --- 4. 에이전트 및 노드(Node) 정의 ---

def triage_agent(state: AgentState):
    print("--- 노드 실행: 접수 담당 에이전트 ---")
    user_input = state["messages"][-1].content
    prompt = f"""당신은 상담 요청을 분류하는 접수 담당자입니다. 사용자의 다음 메시지를 읽고, 가장 관련성이 높은 주제를 '진로', '관계', '스트레스' 중 하나로 분류해주세요. 다른 말은 절대 하지 말고 오직 주제 단어 하나만 출력해야 합니다. 사용자 메시지: "{user_input}" 주제:"""
    response = llm.invoke(prompt)
    topic = response.content.strip()
    ai_message = AIMessage(content=f"네, 고객님의 고민 주제를 '{topic}'으로 파악했습니다. 해당 분야 전문 상담사에게 연결해 드리겠습니다.")
    return {"topic": topic, "messages": [ai_message]}

def specialist_agent(state: AgentState):
    print(f"--- 노드 실행: 전문 상담사 ({state['topic']}) ---")
    topic = state['topic']
    messages = state['messages']

    if "진로" in topic:
        prompt = f"당신은 친절하고 공감 능력이 뛰어난 진로 상담사입니다. 사용자의 고민을 듣고 따뜻한 위로와 함께 현실적인 조언을 해주세요. 대화 기록:\n{messages}\n\n상담사의 답변:"
    elif "관계" in topic:
        prompt = f"당신은 인간관계 전문가이자 따뜻한 상담사입니다. 사용자의 관계 문제를 깊이 이해하고, 관계 개선을 위한 구체적인 조언을 제공해주세요. 대화 기록:\n{messages}\n\n상담사의 답변:"
    else: # 스트레스
        prompt = f"당신은 마음의 안정을 찾아주는 스트레스 관리 전문가입니다. 사용자의 불안과 스트레스 원인을 파악하고, 마음을 진정시킬 수 있는 방법이나 생각의 전환을 제안해주세요. 대화 기록:\n{messages}\n\n상담사의 답변:"

    response = llm.invoke(prompt)
    return {"messages": [response]}

#  PDF 보고서 생성 에이전트
def pdf_report_agent(state: AgentState):
    print("--- 노드 실행: PDF 보고서 생성 에이전트 ---")
    # "pdf 저장" 요청 메시지를 제외한 이전 대화 기록을 사용
    messages_for_report = state["messages"][:-1]

    # 대화 내용이 충분한지 확인
    if len(messages_for_report) < 2: # 최소 사용자 질문 1개, AI 답변 1개
        return {"messages": [AIMessage(content="죄송하지만, 보고서를 생성하기에는 아직 상담 내용이 부족합니다. 상담을 더 진행해주세요.")]}

    prompt = f"""당신은 내담자의 상담 기록을 분석하여 공식적인 보고서를 작성하는 임상 심리 전문가입니다.
아래의 전체 상담 기록을 검토하고, 다음 형식에 맞춰 전문적인 소견 보고서를 작성해주세요.

### 상담 보고서 ###
**1. 내담자 주요 호소 문제:** (핵심 문제 요약)
**2. 상담 과정 요약:** (상담 진행 과정 기술)
**3. 전문가 소견 및 평가:** (전문가적 분석)
**4. 권고 사항:** (구체적인 제안)

---
**상담 기록:**
{messages_for_report}
---
"""
    report_text = llm.invoke(prompt).content
    print("   > 보고서 내용 생성 완료")

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    filename = f"상담_보고서_{timestamp}.pdf"
    create_pdf_report(report_text, filename)

    final_message = f"네, 요청하신 대로 전체 상담 내용을 바탕으로 전문가 보고서를 생성하여 '{filename}' 파일로 저장했습니다."

    return {"report_content": report_text, "messages": [AIMessage(content=final_message)]}

In [ ]:
# --- 5. 조건부 라우팅 함수 정의 ---
def route_logic(state: AgentState):
    """사용자 입력에 따라 상담을 시작할지, PDF를 생성할지 결정합니다."""
    print("--- 라우팅 결정 ---")
    last_message = state["messages"][-1].content.lower()
    if "pdf" in last_message or "저장" in last_message:
        print("   > 경로: PDF 생성")
        return "pdf_reporter"
    else:
        print("   > 경로: 일반 상담")
        return "triage"

In [ ]:
# --- 6. 그래프 생성 및 컴파일 ---
workflow = StateGraph(AgentState)

# 노드 추가
workflow.add_node("triage", triage_agent)
workflow.add_node("specialist", specialist_agent)
workflow.add_node("pdf_reporter", pdf_report_agent)

# 조건부 진입점 설정
# 그래프의 시작점에서 route_logic 함수를 호출하여 첫 노드를 결정
workflow.set_conditional_entry_point(
    route_logic,
    {
        "triage": "triage",
        "pdf_reporter": "pdf_reporter",
    }
)

# 엣지 연결
workflow.add_edge("triage", "specialist")

# 각 분기의 끝을 명시적으로 지정
workflow.add_edge("specialist", END)
workflow.add_edge("pdf_reporter", END)

app = workflow.compile()  # 그래프(워크플로우) 컴파일

In [ ]:
# --- 7. Gradio 인터페이스 정의 (설명 업데이트) ---
def run_counseling_session_gradio(message, history):
    langchain_messages = []
    for human, ai in history:
        langchain_messages.append(HumanMessage(content=human))   # 과거의 사용자 메시지
        langchain_messages.append(AIMessage(content=ai))  # 과거의 챗봇 응답
    # 'history' 처리가 모두 끝난 후, 'message'(방금 입력한 현재 메시지)를 리스트의 맨 마지막에 추가합니다.
    langchain_messages.append(HumanMessage(content=message))

    initial_state = {"messages": langchain_messages}

    # 스트리밍 방식은 여러 노드를 순차 실행할 때 유용하므로,
    # 여기서는 최종 결과만 보여주는 invoke를 사용해도 무방합니다.
    # 하지만 사용자 경험을 위해 스트리밍을 유지합니다.

    final_bot_message = ""
    for chunk in app.stream(initial_state):
        node_name = list(chunk.keys())[0]
        # 해당 노드에서 생성된 최신 AI 메시지를 가져옵니다.
        new_ai_message = chunk[node_name]["messages"][-1]
        final_bot_message += new_ai_message.content + "\n"

    # 마지막에 불필요한 개행문자 제거
    return final_bot_message.strip()

In [ ]:
with gr.Blocks(theme=gr.themes.Soft(), title="LangGraph 상담 챗봇") as demo:
    gr.Markdown(
        """
        # LangGraph 멀티 에이전트 상담 챗봇 (PDF 보고서 기능)

        AI 에이전트 팀이 고민 상담을 진행합니다.
        - 먼저 고민 내용을 자유롭게 입력하여 상담을 시작하세요.
        - 상담이 충분히 진행되었다고 생각되면, **"pdf로 저장해줘"** 또는 **"보고서 저장"** 이라고 입력해보세요.
        - AI가 전체 대화 내용을 분석하여 전문가 소견이 담긴 PDF 보고서를 **자동으로 생성하고 이 프로그램이 실행된 폴더에 저장**해줍니다.
        """
    )
    gr.ChatInterface(
        fn=run_counseling_session_gradio,
        examples=[
            ["요즘 회사에서 앞으로 제 커리어가 어떻게 될지 너무 막막하고 불안해요. 이직을 해야 할까요?"],
            ["가장 친한 친구랑 사소한 일로 다툰 뒤로 계속 어색한데, 어떻게 화해해야 할지 모르겠어요."],
        ],
        title="마음 상담소"
    )

c:\99.참고자료\04.교육\인공지능심화과정(20250721)\소스코드\.venv\lib\site-packages\gradio\chat_interface.py:345: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


In [ ]:
demo.launch()

* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.


In [ ]:
demo.close()

Closing server running on port: 7862
